[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/quickstart/quickstart.ipynb)

# DecimalAI Quickstart

**Get your first traces and see the version-aware loop in 5 minutes.**

This notebook walks you through DecimalAI's core workflow:

1. ✅ Instrument an agent with 2 lines of code
2. ✅ Generate traces (no LLM API key needed)
3. ✅ Change the agent → automatic manifest versioning
4. ✅ See the impact report (keep / repair / replay / drop)

> **No LLM API key required.** This example uses mock tool functions to demonstrate the full workflow.
> For framework-specific examples with real LLMs, see the
> [LangChain quickstart](./quickstart_langchain.ipynb) or
> [OpenAI Agents quickstart](./quickstart_openai_agents.ipynb).

## Step 1 — Install & Configure

In [ ]:
# Install the DecimalAI SDK
!pip install -q decimalai

In [ ]:
import os

# Set your API key (get one at https://app.decimal.ai/settings)
os.environ["DECIMAL_API_KEY"] = "dai_sk_..."  # ← Replace with your key

# Initialize the SDK
import decimalai
decimalai.init()

## Step 2 — Define Your Agent (v1)

We'll create a simple customer support agent with two tools:
- `search_docs` — searches a knowledge base
- `check_inventory` — checks product stock levels

These are mock functions (no real API calls), but DecimalAI traces them
the same way it would trace real tools.

In [ ]:
# --- Agent v1: Two tools ---

def search_docs(query: str) -> str:
    """Search the knowledge base for relevant articles."""
    responses = {
        "password": "To reset your password, go to Settings > Security > Reset Password.",
        "return": "Our return policy allows returns within 30 days of purchase.",
        "shipping": "Standard shipping takes 5-7 business days. Express: 1-2 days.",
        "support": "Contact support at help@example.com or call 1-800-555-0123.",
    }
    for keyword, response in responses.items():
        if keyword in query.lower():
            return response
    return f"Found 3 articles related to '{query}'. Please be more specific."


def check_inventory(product_id: str) -> dict:
    """Check product stock levels."""
    inventory = {
        "SKU-1234": {"product_id": "SKU-1234", "name": "Wireless Mouse", "in_stock": True, "quantity": 42},
        "SKU-5678": {"product_id": "SKU-5678", "name": "USB-C Hub", "in_stock": False, "quantity": 0},
    }
    return inventory.get(product_id, {"product_id": product_id, "in_stock": True, "quantity": 100})


print("✅ Agent v1 defined with 2 tools: search_docs, check_inventory")

## Step 3 — Instrument & Generate Traces

The `@decimalai.trace()` decorator captures everything — inputs, outputs,
tool calls, and timing — and sends it to your dashboard automatically.

In [ ]:
@decimalai.trace(agent_name="support-agent")
def run_agent_v1(query: str) -> str:
    """Simple support agent that routes to the right tool."""
    if "stock" in query.lower() or "inventory" in query.lower():
        result = check_inventory("SKU-1234")
        decimalai.log_tool_call(name="check_inventory", input={"product_id": "SKU-1234"}, output=result)
        return f"Stock check: {result['name']} — {'In stock' if result['in_stock'] else 'Out of stock'} ({result['quantity']} units)"
    else:
        result = search_docs(query)
        decimalai.log_tool_call(name="search_docs", input={"query": query}, output=result)
        return f"Here's what I found: {result}"


# Run 5 queries to generate traces
queries = [
    "How do I reset my password?",
    "Check inventory for SKU-1234",
    "What is your return policy?",
    "Is product SKU-5678 in stock?",
    "How do I contact support?",
]

print("🚀 Running 5 queries through Agent v1...\n")
for q in queries:
    answer = run_agent_v1(q)
    print(f"  Q: {q}")
    print(f"  A: {answer}\n")

print("✅ 5 traces sent to DecimalAI!")
print("📊 Open your dashboard: https://app.decimal.ai/traces")

## Step 4 — Update the Agent (v2)

Now let's simulate a real-world agent update:

1. **Rename** `check_inventory` → `lookup_stock` (clearer name)
2. **Add** a new tool: `process_refund`

This is exactly what happens when your team iterates on an agent.

In [ ]:
# --- Agent v2: Renamed tool + new tool ---

def lookup_stock(item_id: str) -> dict:
    """Look up current stock for an item. (Renamed from check_inventory)"""
    inventory = {
        "SKU-1234": {"item_id": "SKU-1234", "name": "Wireless Mouse", "in_stock": True, "quantity": 42},
        "SKU-5678": {"item_id": "SKU-5678", "name": "USB-C Hub", "in_stock": False, "quantity": 0},
    }
    return inventory.get(item_id, {"item_id": item_id, "in_stock": True, "quantity": 100})


def process_refund(order_id: str, reason: str) -> dict:
    """Process a refund for an order. (NEW in v2)"""
    return {"order_id": order_id, "status": "refunded", "reason": reason, "amount": 29.99}


print("✅ Agent v2 defined:")
print("   - search_docs (unchanged)")
print("   - lookup_stock (renamed from check_inventory)")
print("   - process_refund (NEW)")

In [ ]:
@decimalai.trace(agent_name="support-agent")
def run_agent_v2(query: str) -> str:
    """Updated support agent with renamed + new tools."""
    if "stock" in query.lower() or "inventory" in query.lower():
        result = lookup_stock("SKU-1234")
        decimalai.log_tool_call(name="lookup_stock", input={"item_id": "SKU-1234"}, output=result)
        return f"Stock check: {result['name']} — {'In stock' if result['in_stock'] else 'Out of stock'}"
    elif "refund" in query.lower() or "return" in query.lower():
        result = process_refund("ORD-001", "damaged item")
        decimalai.log_tool_call(name="process_refund", input={"order_id": "ORD-001"}, output=result)
        return f"Refund processed: ${result['amount']} for order {result['order_id']}"
    else:
        result = search_docs(query)
        decimalai.log_tool_call(name="search_docs", input={"query": query}, output=result)
        return f"Here's what I found: {result}"


# Run the updated agent — this triggers a new manifest version
print("🚀 Running Agent v2 (triggers manifest change)...\n")
answer = run_agent_v2("I need to return a damaged item")
print(f"  Q: I need to return a damaged item")
print(f"  A: {answer}\n")

print("✅ Manifest v2 auto-detected! DecimalAI saw the tool changes:")
print("   - check_inventory → renamed to lookup_stock")
print("   - process_refund → added")

## Step 5 — See the Impact Report

Open your **[DecimalAI Dashboard](https://app.decimal.ai)** → **Agents → support-agent**.

You'll see the **Impact Report**:

| Classification | Count | Why |
|---------------|-------|-----|
| **Keep** | 3 | Traces that only used `search_docs` — still compatible |
| **Repair** | 2 | Traces that used `check_inventory` — can be mechanically renamed |
| **Replay** | 0 | No traces need re-running |
| **Drop** | 0 | No tools were removed |

### Why This Matters

If you were fine-tuning on those 5 traces, 2 of them reference a tool
(`check_inventory`) that no longer exists. Training on stale data would teach
your model to call a non-existent tool. **DecimalAI catches this automatically.**

## What Just Happened

```
Agent v1 runs → 5 traces recorded
    ↓
Agent changes (tool renamed, tool added)
    ↓
DecimalAI auto-detects v2 (manifest versioning)
    ↓
Impact Report: 5 traces classified (keep / repair / replay / drop)
    ↓
One-click: Repair stale traces + Build clean dataset
```

This happens **automatically, every time your agent changes.**

## Next Steps

- 📖 [LangChain Quickstart](./quickstart_langchain.ipynb) — Instrument a real LangChain agent
- 📖 [OpenAI Agents Quickstart](./quickstart_openai_agents.ipynb) — Instrument an OpenAI Agents app
- 🔗 [Concepts](https://docs.decimal.ai/concepts) — Understand manifests, traces, and datasets
- 📊 [Dashboard](https://app.decimal.ai) — Explore your traces